# 02. Middle of a list

**Tier:** Deep. **Why you'd use this:** You need the middle value of a linked list, but you do not know the length up front. Counting once then walking halfway is two passes. Two references at different speeds find the middle in one pass.

**After this lesson you can:** advance a slow pointer one step and a fast pointer two steps; read the middle when the fast pointer runs out.
About 25 minutes. **Uses:** [Linked lists](01-traversal-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. How do you move from one node to the next?
2. What value marks the end of a singly linked list?

<details><summary>Answers</summary>

Follow the `nxt` (or `next`) attribute. The end is `None`. Skim [Linked lists](01-traversal-lesson.ipynb) if building a short chain is unclear.
</details>


## Recognize it

You have the head of a singly linked list and need the middle node's value. On an even length, the prompt picks the second middle. Signals: "middle of the linked list", "tortoise and hare", or "slow and fast pointers".

A look-alike that is not this pattern: middle index of a Python list — that is `values[len(values) // 2]` with random access.


## Why it works this way

One honest approach counts nodes in a first pass, then walks `count // 2` steps. That is correct and linear, but touches every node twice. The slow/fast method keeps two references. Each loop iteration moves `slow` by one link and `fast` by two. When `fast` can no longer take a two-step move, `slow` sits on the middle.

For an odd-length list, `fast` lands on the last node and stops; `slow` is on the center. For an even-length list, `fast` becomes `None` after stepping off the end; `slow` sits on the second of the two middle nodes if you move `slow` before checking that `fast` still has a next — which matches the usual "second middle" rule when the loop condition is `fast is not None and fast.nxt is not None` and both pointers start at the head.

The invariant: after `k` iterations, `slow` has moved `k` steps and `fast` has moved `2k` steps (or hit the end). The loop ends when `2k` reaches or passes the length, so `k` is about half the length. Time is still proportional to `n` because `fast` walks about `n` links. Extra memory is two pointers — constant. Cycle detection uses the same two speeds with a different stop rule; here the list is assumed acyclic.
Splitting a list for merge sort uses the same slow/fast pass: when fast finishes, slow is at the midpoint and `slow.nxt` starts the right half (after you cut the link). Palindrome checks reverse the second half after finding the middle, then compare. Cycle detection is the other famous two-speed walk: if fast ever meets slow again after leaving the head, a cycle exists.

Starting both pointers at the head is required for the second-middle convention on even lengths. Starting slow one step ahead changes which middle you land on. Read the prompt's even-length rule before you write the loop condition. Empty lists are out of scope here; a single-node list returns that node's value immediately because the loop never runs.


## When to use it

Use slow/fast when you need the middle (or to split a list) in one pass with constant extra memory. Prefer a count-then-walk when clarity matters more than the second pass. Prefer an array index when the data is already a Python list.
 When the language exposes a size field on the list object, using that length is fine — challenge prompts usually hide it to force the pointer walk.


## Brute force, and why it's slow

Count the length, then walk halfway. Correct, but every node is visited twice.


In [1]:
class Node:
    def __init__(self, value, nxt=None):
        self.value = value
        self.nxt = nxt


def middle_value_slow(head):
    visits = 0
    count = 0
    node = head
    while node is not None:
        count += 1
        visits += 1
        node = node.nxt
    node = head
    for _ in range(count // 2):
        visits += 1
        node = node.nxt
    return node.value, visits


def chain(values):
    head = None
    for value in reversed(values):
        head = Node(value, head)
    return head


for size in (51, 101, 201):
    print(size, middle_value_slow(chain(list(range(size)))))


51 (25, 76)
101 (50, 151)
201 (100, 301)


## Key insight

Move one pointer a single step and another pointer two steps in the same loop. When the fast pointer cannot take another double step, the slow pointer is on the middle.


## Worked example

Return the middle value. On an even number of nodes, return the second middle.


In [2]:

def middle_value(head) -> int:
    # 1. Both start at the head.
    slow = fast = head
    # 2. Fast takes two steps; slow takes one.
    while fast is not None and fast.nxt is not None:
        slow = slow.nxt
        fast = fast.nxt.nxt
    # 3. Slow lands on the middle (second middle if even).
    return slow.value


print(middle_value(Node(1, Node(2, Node(3)))))
print(middle_value(Node(1, Node(2, Node(3, Node(4))))))
print(middle_value(Node(1)))


2
3
1


## Trace

`1 → 2 → 3 → 4`:

| step | slow | fast |
|---|---|---|
| start | 1 | 1 |
| 1 | 2 | 3 |
| 2 | 3 | `None` (after 3 → 4 → None) |

Loop stops; return **3** (second middle).


## Cost

`fast` walks about `n` links once: time grows with `n`. Extra memory is two pointers. The count-then-walk method is also linear but roughly doubles the visits.

Constant extra memory matters in tight environments where allocating a second array of values is unacceptable. The two-pointer walk never materializes the list as a Python `list`; it only follows existing links. That is the design reason to prefer slow/fast once the node API is given.


## Variants

Not solved here: [Linked List Cycle](../../challenges/problems/24-linked-list-cycle.ipynb), [Remove Nth Node From End of List](../../challenges/problems/56-remove-nth-from-end.ipynb), [Reorder List](../../challenges/problems/60-reorder-list.ipynb).


## Common mistake

This bug is silent. The loop moves only `fast` by two and forgets to advance `slow`, so the function always returns the head's value. There is no traceback. The print is `1`, and the answer is `3`.


In [3]:
def middle_value_wrong(head):
    slow = fast = head
    while fast is not None and fast.nxt is not None:
        fast = fast.nxt.nxt  # slow never moves
    return slow.value


print(middle_value_wrong(Node(1, Node(2, Node(3, Node(4, Node(5)))))))


1


The fix advances `slow = slow.nxt` in the same loop body. Run `middle_value` on `1→2→3→4→5` and you get `3`.


In [4]:
print(middle_value(Node(1, Node(2, Node(3, Node(4, Node(5)))))))


3


## Python idioms

Tuple assignment updates both pointers in one line: `slow, fast = slow.nxt, fast.nxt.nxt`. The expanded form is clearer while you learn the invariant.


In [5]:
def middle_value_tuple(head) -> int:
    slow = fast = head
    while fast is not None and fast.nxt is not None:
        slow, fast = slow.nxt, fast.nxt.nxt
    return slow.value


print(middle_value_tuple(Node(1, Node(2, Node(3)))))


2


## Recap

- Slow moves one link; fast moves two.
- When fast cannot continue, slow is at the middle.
- Even length: this loop returns the second middle.

Next: [practice](02-middle-practice.ipynb).
